# Урок 33 — Multiprocessing: Справжній Паралелізм та Executor Architecture

## Ізоляція пам'яті, GIL, IPC, ProcessPool та concurrent.futures

**Модуль 4 · Network & Concurrent Systems · Автор: Viktor Nikoriak**

---

# Про що цей урок?

У попередньому уроці ми вивчили `threading` та зіткнулись з **GIL**.

Тепер питання: *що робити з CPU-bound задачами, де GIL блокує паралелізм?*

Відповідь: **`multiprocessing`** — Python запитує ОС клонувати весь інтерпретатор.

Але ось що дивує більшість розробників:

```python
counter = 0

def worker():
    global counter
    counter += 1

p = multiprocessing.Process(target=worker)
p.start(); p.join()

print(counter)  # Що виведе?
```

Якщо ти відповів `1` — цей урок саме для тебе.

---

**Цей урок розбиває 5 хибних уявлень:**

| # | Хибне уявлення | Реальність |
|---|----------------|------------|
| 1 | Процеси та потоки — одне й те саме | Повна ізоляція пам'яті через fork()/spawn() |
| 2 | `threading` прискорює CPU-задачі | GIL перетворює це на черговість + overhead |
| 3 | `multiprocessing` завжди швидший | Серіалізація (pickle) може знищити продуктивність |
| 4 | Executor — це просто обгортка | Це архітектура "менеджер + пул" |
| 5 | Future дозволяє async | `.result()` блокує повністю |

# Learning Objectives

Після цього уроку ти зможеш:

## Conceptual Understanding

- Пояснити різницю між **Thread** та **Process** на рівні адресних просторів ОС
- Описати що відбувається коли Python викликає `fork()` vs `spawn()`
- Пояснити чому `multiprocessing` **не вирішує** проблему спільної пам'яті
- Описати механізм **IPC (Inter-Process Communication)** через OS Pipe
- Пояснити коли `multiprocessing` **уповільнює** програму

## Debugging Skills

- Передбачити вивід коду з ізоляцією пам'яті
- Визначити коли `pickle` призводить до `TypeError`
- Розпізнати Zombie Process та CPU Oversubscription
- Знайти причину `RuntimeError` при запуску на Windows

## Production Skills

- Правильно використовувати `ProcessPoolExecutor` для CPU-bound задач
- Використовувати `ThreadPoolExecutor` для I/O-bound задач
- Розуміти `Future` об'єкти та `as_completed()`
- Писати безпечний код з `if __name__ == '__main__':` guard

# Ментальна модель: Клітинний поділ

Щоб зрозуміти `multiprocessing`, потрібно **перестати думати** про свою програму як про єдину сутність.

Натомість уяви свій Python-код як **біологічну клітину**, яка ділиться (мітоз).

```
До fork():                      Після fork():

  +─────────────────+           +──────────────────+    +──────────────────+
  │  Python Process │           │ Parent (PID 1000) │    │  Child (PID 1001)│
  │                 │  fork()   │                  │    │                  │
  │  counter = 0    │ ────────> │  counter = 0     │    │  counter = 0     │
  │  GIL            │           │  Own GIL         │    │  Own GIL (copy!) │
  │  Heap           │           │  Heap (original) │    │  Heap (isolated!)│
  +─────────────────+           +──────────────────+    +──────────────────+
                                      CPU Core 1              CPU Core 2
```

**Ключова ідея:** Після `fork()` ці два процеси **ніколи більше не діляться пам'яттю**.
Зміна в дочірньому процесі = зміна на іншому комп'ютері.

---

# Вправа 1 — Ілюзія спільного світу

## Неправильне припущення

> Потоки і процеси — це одне й те саме, тільки процеси "потужніші"

## Передбач результат

Подивись на код нижче. **Не запускай.** Що виведе кожен сценарій?

- А) Обидва сценарії додадуть елемент до списку
- Б) Процес виведе порожній список `[]`
- В) Програма аварійно завершиться

In [ ]:
import threading
import multiprocessing
import os

shared_data = []

def add_data(item):
    shared_data.append(item)
    print(f"  Всередині функції: {shared_data}")

print("=" * 50)
print("Сценарій A: Threading (спільна пам'ять)")
print("=" * 50)

t1 = threading.Thread(target=add_data, args=('Thread-A',))
t1.start()
t1.join()
print(f"Після потоку: {shared_data}")

print()
print("=" * 50)
print("Сценарій B: Multiprocessing (ізольована пам'ять)")
print("=" * 50)

shared_data = []  # Очищаємо

if __name__ == '__main__':
    p1 = multiprocessing.Process(target=add_data, args=('Process-A',))
    p1.start()
    p1.join()
    print(f"Після процесу: {shared_data}")

## Що відбулось?

**Результат:**
```
Сценарій A: Threading
  Всередині функції: ['Thread-A']
Після потоку: ['Thread-A']          <-- Зміна видима

Сценарій B: Multiprocessing
  Всередині функції: ['Process-A']   <-- Виконується в ІНШОМУ процесі
Після процесу: []                    <-- Оригінал НЕ змінився
```

## Покрокова шкала виконання

| Час | Threading | Multiprocessing |
|-----|-----------|----------------|
| T=0 | Потік запускається | ОС виконує fork() — клонує всю пам'ять |
| T=1 | Доступ до спільного shared_data у Heap | Дочірній процес має **свою копію** |
| T=2 | Додає елемент до оригіналу | Додає до **своєї** копії |
| T=3 | Потік завершується | Дочірній процес завершується, пам'ять знищується |
| T=4 | Батьківський бачить зміни | Батьківський бачить незмінений оригінал |

## Аналогія

> **Threading:** Колеги в одному кабінеті пишуть на **одній** маркерній дошці.
> **Multiprocessing:** Ти копіюєш дошку, несеш копію в інший кабінет. Там малюють на копії, твоя залишається чистою.

## Debugging Tip

```python
import os
print(f"PID: {os.getpid()}")  # У батька і дитини різні PID
```

---

# Вправа 2 — Парадокс уповільнення: GIL та CPU-задачі

## Неправильне припущення

> Додавання `threading` до важких обчислень зробить їх швидшими

## Передбач результат

Що відбудеться з часом виконання порівняно з послідовним запуском?

- А) Потоки і процеси — вдвічі швидше
- Б) Потоки швидші, бо вони легші
- В) Процеси вдвічі швидші, а потоки — **повільніші** ніж без потоків

In [ ]:
import threading
import multiprocessing
import time

def heavy_math():
    return sum(i * i for i in range(5_000_000))

# Базовий рівень: послідовно
start = time.perf_counter()
heavy_math()
heavy_math()
sequential_time = time.perf_counter() - start
print(f"Послідовно:        {sequential_time:.2f}s")

# Threading
start = time.perf_counter()
t1 = threading.Thread(target=heavy_math)
t2 = threading.Thread(target=heavy_math)
t1.start(); t2.start()
t1.join(); t2.join()
threading_time = time.perf_counter() - start
print(f"Threading:         {threading_time:.2f}s  (x{threading_time/sequential_time:.1f})")

# Multiprocessing
if __name__ == '__main__':
    start = time.perf_counter()
    p1 = multiprocessing.Process(target=heavy_math)
    p2 = multiprocessing.Process(target=heavy_math)
    p1.start(); p2.start()
    p1.join(); p2.join()
    mp_time = time.perf_counter() - start
    print(f"Multiprocessing:   {mp_time:.2f}s  (x{mp_time/sequential_time:.1f})")

    if threading_time > sequential_time:
        print()
        print("Threading ПОВIЛЬНIШИЙ за послiдовний — GIL у дiї!")
    if mp_time < sequential_time:
        print("Multiprocessing ШВИДШИЙ — кожен процес має власний GIL!")

## Чому threading уповільнює CPU-задачі?

```
Threading (один GIL на двох потоків):

CPU Core 1: [Thread1 ##........##........##]
CPU Core 2: [........Thread2..........Thread2]
                  ^
            Context Switch (переключення контексту — дорога операція!)

Multiprocessing (кожен процес має власний GIL):

CPU Core 1: [Process1 ####################]
CPU Core 2: [Process2 ####################]
                  ^
            Справжнiй паралелiзм!
```

## GIL: Чому він існує?

**GIL (Global Interpreter Lock)** — м'ютекс, що захищає внутрішні структури CPython.

Python управляє пам'яттю через **reference counting**. Без GIL два потоки могли б одночасно декрементувати лічильник об'єкта — segfault.

## Правило вибору

| Тип задачі | Приклади | Інструмент |
|------------|----------|-----------|
| I/O-bound | HTTP запити, читання файлів, БД | `threading` / `asyncio` |
| CPU-bound | математика, ML inference, обробка зображень | `multiprocessing` |

---

# Вправа 3 — Невидима стіна: Вартість Серіалізації

## Неправильне припущення

> Якщо multiprocessing швидший для математики — використовуй завжди

## Передбач результат

```python
huge_dataset = [list(range(10000)) for _ in range(500)]

def process_chunk(data_chunk):
    return sum(data_chunk)
```

Що буде швидше: ThreadPool чи ProcessPool?

- А) ProcessPool — математична задача, буде швидший
- Б) ThreadPool — менший overhead
- В) Однаково

In [ ]:
import sys, os, pathlib, importlib.util

# Find the exact on-disk location of _note_workers.py regardless of Jupyter CWD.
# Workers spawn a fresh interpreter — they inherit os.environ (OS-level) but
# sys.path propagation via multiprocessing is unreliable on Windows/Python 3.14.
# Setting PYTHONPATH guarantees workers find the module before any Python code runs.
_spec = importlib.util.find_spec('_note_workers')
if _spec and _spec.origin:
    _lesson_dir = os.path.dirname(os.path.abspath(_spec.origin))
else:
    _lesson_dir = str(pathlib.Path('_note_workers.py').resolve().parent)

if _lesson_dir not in sys.path:
    sys.path.insert(0, _lesson_dir)

_pp = os.environ.get('PYTHONPATH', '')
if _lesson_dir not in _pp:
    os.environ['PYTHONPATH'] = _lesson_dir + (os.pathsep + _pp if _pp else '')

from _note_workers import (
    process_chunk,
    heavy_computation,
    intense_math,
    process_image_chunk,
    convert_grayscale,
    worker_task,
)
print(f'_note_workers imported OK')
print(f'Workers will find it at: {_lesson_dir}')
print(f'PYTHONPATH: {os.environ["PYTHONPATH"][:80]}')

In [ ]:
# process_chunk iмпортовано з _note_workers.py
import concurrent.futures
import time

# 200 chunks x 2,000 items -- демонструє IPC overhead без перевантаження Windows pipe
dataset = [list(range(2_000)) for _ in range(200)]

if __name__ == '__main__':
    start = time.perf_counter()
    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
        results_t = list(executor.map(process_chunk, dataset))
    thread_time = time.perf_counter() - start
    print(f'ThreadPool:   {thread_time:.3f}s')

    start = time.perf_counter()
    with concurrent.futures.ProcessPoolExecutor(max_workers=4) as executor:
        results_p = list(executor.map(process_chunk, dataset))
    process_time = time.perf_counter() - start
    print(f'ProcessPool:  {process_time:.3f}s')

    ratio = process_time / thread_time
    verdict = 'швидший' if ratio < 1 else 'повiльнiший'
    print(f'ProcessPool був {verdict} у {ratio:.1f}x')
    print()
    print('Чому? ProcessPool pickle-ує кожен список через OS Pipe.')
    print('Передача даних коштує бiльше нiж самi обчислення (sum).')

## IPC Pipeline: Як дані подорожують між процесами

```
PRODUCER PROCESS (User Space)            CONSUMER PROCESS (User Space)
+──────────────────────────+            +──────────────────────────+
│ 1. data = [0..9999]      │            │ 4. data = [0..9999]      │
│ 2. pickle.dumps(data)    │            │ 3. pickle.loads(bytes)   │
│    -> ~80 KB байтiв      │            │    <- ~80 KB байтiв      │
+────────────+─────────────+            +────────────^─────────────+
             │                                        │
═════════════╪════════════════════════════════════════╪══════════════
                              OS KERNEL SPACE
                    [ Anonymous Pipe Buffer ] ──────>
═════════════════════════════════════════════════════════════════════
```

## Правило архітектури

> **Передавай якомога менше.** Ідеальний сценарій: передай невелике ім'я файлу або chunk ID — нехай процес сам читає дані з диска у своїй ізольованій пам'яті.

```python
# Погано: передаємо гiгабайти через IPC
with ProcessPoolExecutor() as ex:
    results = ex.map(process, [huge_array] * 100)

# Добре: передаємо лише шляхи, процес читає сам
with ProcessPoolExecutor() as ex:
    results = ex.map(process_file, ["/data/chunk_01.bin", "/data/chunk_02.bin"])
```

---

# Глибоко під капотом: Рівень Операційної Системи

## fork() vs spawn()

| | Unix/macOS | Windows |
|--|------------|---------|
| Механізм | fork() | spawn() |
| Швидкість | Миттєво (Copy-on-Write) | Повiльно (новий iнтерпретатор) |
| Пам'ять | Клон поточного стану | Чистий Python, потiм pickle |
| Небезпека | Дублює вiдкритi socket-и | Рекурсивний import без guard |

In [ ]:
import multiprocessing as mp
import os

counter = 0

def worker():
    global counter
    counter += 1
    # PID показує, що ми — окремий процес ОС
    print(f"  Child  [PID: {os.getpid()}] counter = {counter}")

if __name__ == '__main__':
    print(f"Parent [PID: {os.getpid()}] counter = {counter}")

    p = mp.Process(target=worker)

    # p.start() -> ОС виконує fork():
    # - Батьківський процес зупиняється на мить
    # - ОС копіює весь адресний простір (Copy-on-Write)
    # - Дочірній процес починає виконання
    p.start()

    # join() -> батько робить blocking system call
    # ОС переводить батька в сон (0% CPU) до завершення дитини
    p.join()

    # Оригінальний counter НЕ змінився!
    print(f"Parent [PID: {os.getpid()}] counter = {counter}")
    print()
    print("Два рiзних PID = два iзольованi адреснi простори ОС")

## IPC через multiprocessing.Queue

Оскільки процеси ізольовані, вони спілкуються через **OS-рівневі труби**.

`multiprocessing.Queue` — це не простий масив у пам'яті.
Під капотом Python просить ОС створити **anonymous C-level pipe** в Kernel Space.

**Кроки передачі:**
1. Producer: `pickle.dumps(obj)` — об'єкт стає байтовим потоком
2. Байти записуються в OS pipe (Kernel Space)
3. Consumer: `q.get()` — blocking I/O, ОС переводить процес у сон
4. ОС прокидає Consumer: `pickle.loads(bytes)` — байти стають Python-об'єктом

In [ ]:
import multiprocessing as mp
import pickle

def producer(q):
    data = {"task_id": 42, "status": "pending", "payload": list(range(100))}

    # Що відбувається під капотом:
    # 1. pickle.dumps(data) -> байтовий потiк
    # 2. bytes записуються в OS pipe (Kernel Space)
    q.put(data)
    print(f"  Producer: вiдправив {len(pickle.dumps(data))} байт через OS pipe")

def consumer(q):
    # 1. q.get() - blocking I/O call
    # 2. ОС переводить процес у сон, чекає на байти в pipe
    # 3. Отримує байти -> pickle.loads() -> Python dict
    item = q.get()
    print(f"  Consumer: отримав task_id={item['task_id']}, payload len={len(item['payload'])}")

if __name__ == '__main__':
    q = mp.Queue()

    p1 = mp.Process(target=producer, args=(q,))
    p2 = mp.Process(target=consumer, args=(q,))

    p1.start(); p2.start()
    p1.join(); p2.join()

    print("Серiалiзацiя (pickle) — єдиний спосiб передати Python-об'єкт мiж процесами")

## ProcessPoolExecutor: Lifecycle та CPU Core Utilization

```
[ MAIN PARENT PROCESS ]
      |
      | ProcessPoolExecutor(max_workers=4)
      | -> ОС створює рівно 4 worker-процеси
      | -> Кожен входить у while True: очікування з pipe
      |
      +-> [ CORE 1 ] Worker Process 1  (tasks 0, 4, 8)
      +-> [ CORE 2 ] Worker Process 2  (tasks 1, 5, 9)
      +-> [ CORE 3 ] Worker Process 3  (tasks 2, 6)
      +-> [ CORE 4 ] Worker Process 4  (tasks 3, 7)

Після виходу з `with` блоку:
-> Parent надсилає "poison pill" (sentinel) у pipe
-> Workers розпаковують sentinel -> добровільно завершуються
-> ОС видаляє їх з таблиці процесів
```

**Zombie Prevention:** якщо батьківський процес падає без `join()` — дочірні стають
**zombies** і займають OS process table нескінченно.

In [ ]:
# heavy_computation iмпортовано з _note_workers.py
import concurrent.futures
import os
import time

if __name__ == '__main__':
    print(f'Parent PID: {os.getpid()}')
    print(f'CPU cores: {os.cpu_count()}')
    print()

    with concurrent.futures.ProcessPoolExecutor(max_workers=4) as executor:
        futures = [executor.submit(heavy_computation, i) for i in range(8)]

        for f in concurrent.futures.as_completed(futures):
            x, result, worker_pid = f.result()
            print(f'  Task {x}: result={result}, Worker PID={worker_pid}')

    print()
    print("Пiсля 'with' -- всi worker-процеси знищенi ОС")

---

# concurrent.futures — Архітектура Executor

## Ментальна модель: Ресторан

```
Стара модель (threading/multiprocessing):     Нова модель (Executor):

Ти особисто:                                  Ти:
1. Наймаєш кухаря                            1. Наймаєш менеджера з командою 3 кухарів
2. Пояснюєш як смажити яйце                 2. Передаєш "тiкет" (задачу) менеджеру
3. Стоїш поряд поки не закінчить            3. Менеджер кладе тiкет у чергу
4. Звільняєш кухаря                         4. Вiльний кухар бере тiкет, готує, дзвонить
```

## ThreadPoolExecutor vs ProcessPoolExecutor

| | ThreadPoolExecutor | ProcessPoolExecutor |
|--|-------------------|---------------------|
| Пам'ять | Спiльна | Iзольована |
| GIL | Один на всiх | Власний у кожного |
| Передача даних | Pointer (миттєво) | pickle через OS pipe |
| Iдеально для | I/O-bound | CPU-bound |
| if __name__ guard | Не потрiбен | **Обов'язковий** на Windows |

## Executor Lifecycle

```
[T=0] Входимо в 'with' -> ОС створює N worker-потокiв/процесiв
[T=1] submit(task) -> задача в чергу, повертає Future негайно
[T=2] Worker бере задачу, виконує
[T=3] Виходимо з 'with' -> shutdown(wait=True) -> чекає всiх worker-iв
[T=4] Всi завершились -> пул знищено
```

In [ ]:
import concurrent.futures
import time

def simulate_io_task(task_id):
    time.sleep(0.3)  # Симуляцiя мережевої затримки
    return f"result_{task_id}"

def simulate_cpu_task(n):
    return sum(i * i for i in range(n * 100_000))

print("I/O-bound задачi (ThreadPool рекомендовано):")
start = time.perf_counter()
with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
    results = list(executor.map(simulate_io_task, range(10)))
t = time.perf_counter() - start
print(f"  ThreadPool: {t:.2f}s для 10 задач по 0.3s (паралельно!)")
print(f"  Очiкували б послiдовно: {10 * 0.3:.1f}s")

print()
print("Lifecycle демо:")
print("  [T=0] Входимо в 'with' -> створюємо пул")

with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
    print("  [T=1] Пул готовий")

    # submit() не блокує! Повертає Future негайно
    f1 = executor.submit(simulate_io_task, 1)
    f2 = executor.submit(simulate_io_task, 2)
    print(f"  [T=2] Двi задачi в черзi. Future статус: done={f1.done()}")

    # as_completed повертає Future в порядку завершення (не submit!)
    for completed in concurrent.futures.as_completed([f1, f2]):
        print(f"  [T=?] Завершено: {completed.result()}")

print("  [T=end] Вийшли з 'with' -> shutdown(wait=True) -> пул знищено")

## Future об'єкти: Ресторанний пейджер

Коли ти `submit()` задачу до Executor — отримуєш `Future`.

**Future** = проксі-об'єкт для обчислення, яке ще не завершилось.

```
States машина Future:

  submit()          worker починає          worker завершує
     |                   |                        |
  [PENDING] -----> [RUNNING] --------------> [FINISHED]
                                               |      |
                                           result  exception
```

### Критичне попередження: Exception Swallowing

```python
future = executor.submit(lambda: 1/0)
time.sleep(2)
print("Програма завершилась!")  # <- Це виведеться!
```

`ZeroDivisionError` **замовчується** всередині Future.
Якщо ти ніколи не викликаєш `future.result()` — помилка зникає назавжди.

### .result() — це блокуючий виклик!

```python
# Неправильно: негайний .result() знищує сенс concurrency
future = executor.submit(heavy_task)
result = future.result()  # Блокує main thread! Нiяк не краще нiж sequential

# Правильно: збирай всi futures, потiм обробляй
futures = [executor.submit(task, i) for i in range(10)]
for f in concurrent.futures.as_completed(futures):
    print(f.result())  # Обробляємо мiру готовностi
```

In [ ]:
import concurrent.futures
import time

def risky_task(x):
    if x == 3:
        raise ValueError(f"Задача {x} зламана!")
    time.sleep(0.1 * x)
    return x * 10

if __name__ == '__main__':
    with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
        futures = {executor.submit(risky_task, i): i for i in range(5)}

        print("Статуси вiдразу пiсля submit:")
        for f, i in futures.items():
            print(f"  task_{i}: done={f.done()}, running={f.running()}")

        print()
        print("Результати в порядку завершення (as_completed):")
        for completed in concurrent.futures.as_completed(futures):
            task_id = futures[completed]
            try:
                result = completed.result()
                print(f"  task_{task_id}: {result}")
            except Exception as e:
                # Exception був "заморожений" у Future пiд час виконання
                print(f"  task_{task_id}: {type(e).__name__}: {e}")

        print()
        print("Правило: ЗАВЖДИ обгортай future.result() в try/except")

## Abstraction vs Control: коли Executor недостатньо

**Executor** — як Uber. Ефективно, але ти не контролюєш маршрут.
**Raw threading/multiprocessing** — як власне авто. Більше роботи, але повний контроль.

```python
# Executor приховує:
# - який саме worker виконує задачу
# - PID worker-процесу
# - як саме спiлкуватись з конкретним worker-ом

# Raw multiprocessing дає:
# - повний контроль над процесами
# - можливiсть сигналiв (SIGTERM, SIGKILL)
# - shared memory та Manager objects
```

### cancel() та його обмеження

```python
future = executor.submit(download_huge_file)

# Якщо задача вже виконується — cancel() НЕ зупинить download!
success = future.cancel()  # True тiльки якщо задача ще в чергу
```

**Правило:** Cancel працює лише для задач ще в черзi. Виконання зупинити не можна.
Для timeout — додавай `timeout=` всередину функцiї (`requests.get(url, timeout=5)`).

In [ ]:
import concurrent.futures
import time

def quick_task(x):
    time.sleep(0.05)
    return x * 2

def slow_task(x):
    time.sleep(5)  # Симуляцiя зависання
    return x * 2

if __name__ == '__main__':
    # Демо: cancel() для задач у черзi
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
        # Перша задача займає worker
        f1 = executor.submit(slow_task, 1)
        time.sleep(0.01)  # Чекаємо щоб f1 почав виконуватись

        # Друга задача в черзi (worker зайнятий)
        f2 = executor.submit(quick_task, 2)

        # Спробуємо скасувати
        cancelled = f2.cancel()
        print(f"f2 cancelled: {cancelled}")
        print(f"f1 cancelled (вже виконується): {f1.cancel()}")

        # f1 все одно виконається (занадто пiзно скасовувати)
        try:
            result = f1.result(timeout=1.0)
        except concurrent.futures.TimeoutError:
            print("f1 timeout (все ще виконується)...")

    print()
    print("Висновок: cancel() тiльки для задач у черзi, не для виконуваних")

---

# Типові помилки та Debugging

## Помилка 1 — Рекурсивна бомба (Windows)

**Симптоми:** На Linux/macOS працює, на Windows — нескiнченнi вiкна або `RuntimeError`.

```python
# Неправильно — без guard
from multiprocessing import Process

def worker():
    print("Worker!")

p = Process(target=worker)
p.start()  # Windows: spawn() -> import script -> ще один Process() -> ...
p.join()
```

**Чому?** Windows не має `fork()`, використовує `spawn()`. Новий iнтерпретатор
**iмпортує** скрипт -> виконує module-level код -> знову `Process()` -> рекурсiя.

```python
# Правильно
if __name__ == '__main__':
    p = Process(target=worker)
    p.start()
    p.join()
```

## Помилка 2 — Pickle Error

Перевiряй **до** передачi в Process:

```python
import pickle

def can_pickle(obj):
    try:
        pickle.dumps(obj)
        return True
    except:
        return False

# socket, lambda, generator, open file -- НЕ МОЖНА передати в Process
```

In [ ]:
import pickle
import socket

def can_pickle(obj):
    try:
        pickle.dumps(obj)
        return True
    except Exception as e:
        return False

print("Що можна передати в Process:")
print(f"  int 42:            {can_pickle(42)}")
print(f"  str 'hello':       {can_pickle('hello')}")
print(f"  list [1,2,3]:      {can_pickle([1, 2, 3])}")
print(f"  dict {{key: val}}:  {can_pickle({'key': 'value'})}")
print(f"  tuple (1,2,3):     {can_pickle((1, 2, 3))}")

print()
print("Що НЕ можна передати в Process:")

s = socket.socket()
print(f"  socket object:     {can_pickle(s)}")
s.close()

lam = lambda x: x * 2
print(f"  lambda function:   {can_pickle(lam)}")

gen = (x for x in range(10))
print(f"  generator:         {can_pickle(gen)}")

print()
print("Аналогiя: можна 'надiслати' ключ вiд кiмнати другу в iнше мiсто?")
print("Нi — лише план ключа (примiтивнi данi), щоб вiн зробив свiй.")

## Помилка 3 — Zombie Processes

Дочiрнiй процес завершується, але батько не викликав `join()`.
ОС зберiгає "надгробок" (exit code) у таблицi процесiв назавжди.

```python
# Неправильно: zombie factory
for _ in range(100):
    p = Process(target=task)
    p.start()
    # Забули: p.join()  ->  100 zombies в таблицi процесiв!

# Правильно: завжди збирай процеси
processes = [Process(target=task) for _ in range(100)]
for p in processes:
    p.start()
for p in processes:
    p.join()  # ОС читає exit code -> видаляє запис
```

## Помилка 4 — CPU Oversubscription

**Симптоми:** Вентилятор ноутбука на максимум, але код ПОВIЛЬНIШИЙ нiж sequential.

```
Маєш 4 cores, запускаєш 100 процесiв:

CPU: [P1-P2-P3-P4-P5-P6-P7-P8-...-P100]
          ^Context Switch^ (дорого!)
          ^Context Switch^ (дорого!)
          ^Context Switch^ (дорого!)

Процесор бiльше перемикається, нiж рахує.
```

**Правило для CPU-bound: max_workers <= os.cpu_count()**

In [ ]:
# intense_math iмпортовано з _note_workers.py
import os
import concurrent.futures
import time

if __name__ == '__main__':
    cpu_count = os.cpu_count()
    WIN_MAX = 61  # Windows: ProcessPoolExecutor hard limit
    print(f'CPU cores: {cpu_count}')
    print()

    for n_workers in [1, cpu_count, min(cpu_count * 4, WIN_MAX)]:
        start = time.perf_counter()
        with concurrent.futures.ProcessPoolExecutor(max_workers=n_workers) as ex:
            list(ex.map(intense_math, range(cpu_count * 2)))
        elapsed = time.perf_counter() - start
        marker = '<-- оптимум' if n_workers == cpu_count else ''
        print(f'  {n_workers:3d} workers -> {elapsed:.3f}s  {marker}')

    print()
    print(f'Оптимум для CPU-bound: max_workers={cpu_count}')
    print('Бiльше нiж cpu_count -> Context Switching знищує прирiст')

---

# Production Usage: Реальнi Патерни

## Патерн 1: I/O-bound Web Scraping (ThreadPool)

**Коли:** HTTP requests, database queries, file reads.
**Правило:** завжди `timeout=` у мережевих запитах.

In [ ]:
import concurrent.futures
import time
import random

def fetch_url(url_id: int) -> dict:
    # Симуляцiя HTTP запиту з можливою помилкою
    latency = random.uniform(0.1, 0.5)
    time.sleep(latency)
    if random.random() < 0.1:
        raise ConnectionError(f"Timeout для URL {url_id}")
    return {"url_id": url_id, "status": 200, "latency_ms": int(latency * 1000)}

def scrape_urls(url_ids: list, max_workers: int = 10) -> tuple:
    results = []
    errors = []

    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_url = {executor.submit(fetch_url, uid): uid for uid in url_ids}

        for future in concurrent.futures.as_completed(future_to_url):
            url_id = future_to_url[future]
            try:
                result = future.result(timeout=2.0)  # Завжди timeout!
                results.append(result)
            except concurrent.futures.TimeoutError:
                errors.append({"url_id": url_id, "error": "timeout"})
            except Exception as e:
                errors.append({"url_id": url_id, "error": str(e)})

    return results, errors

if __name__ == '__main__':
    url_ids = list(range(20))
    start = time.perf_counter()
    results, errors = scrape_urls(url_ids, max_workers=10)
    elapsed = time.perf_counter() - start

    print(f"Успiшно: {len(results)}/20 URL за {elapsed:.2f}s")
    print(f"Помилки: {len(errors)}")
    if results:
        avg = sum(r['latency_ms'] for r in results) / len(results)
        print(f"Середня затримка: {avg:.0f}ms")

## Патерн 2: CPU-bound Image Processing (ProcessPool)

**Коли:** image resizing, ML inference, data compression.
**Правило:** передавай тiльки простi типи (string, int, tuple) через IPC.

In [ ]:
# process_image_chunk iмпортовано з _note_workers.py
import concurrent.futures
import os
import time

if __name__ == '__main__':
    n_workers = os.cpu_count()
    chunks = [(i, 10_000) for i in range(n_workers * 2)]

    print(f'Обробляємо {len(chunks)} chunks на {n_workers} cores...')

    start = time.perf_counter()
    with concurrent.futures.ProcessPoolExecutor(max_workers=n_workers) as executor:
        results = list(executor.map(process_image_chunk, chunks))
    elapsed = time.perf_counter() - start

    unique_workers = len(set(r['worker_pid'] for r in results))
    print(f'Оброблено: {len(results)} chunks за {elapsed:.3f}s')
    print(f'Унiкальнi worker PID: {unique_workers} (= {n_workers} cores)')

---

# Практичнi Вправи

## Вправа 1: Shared Counter (Рiвень: Basic)

Напиши код, де 5 процесiв пробують збiльшити спiльний лiчильник.
Поясни чому результат не буде 5 — i виправ за допомогою `multiprocessing.Value`.

In [ ]:
import multiprocessing as mp

def increment_broken(val_list):
    # Чому це не працює?
    val_list.append(1)

def increment_fixed(shared_val, lock):
    # TODO: реалiзуй правильний варiант
    pass
    # with lock:
    #     shared_val.value += 1

if __name__ == '__main__':
    # Тест зламаного варiанту
    manager = mp.Manager()
    broken_list = manager.list()  # Спiльний список через Manager

    processes = []
    for _ in range(5):
        p = mp.Process(target=increment_broken, args=(broken_list,))
        p.start()
        processes.append(p)
    for p in processes:
        p.join()
    print(f"Manager list: очiкували 5, отримали {len(broken_list)}")

    # TODO: реалiзуй з mp.Value + mp.Lock
    # shared_val = mp.Value('i', 0)
    # lock = mp.Lock()
    # ...
    # print(f"Fixed Value: {shared_val.value}")

## Вправа 2: Правильний вибiр iнструменту (Рiвень: Intermediate)

Для кожної задачi обери: `threading`, `multiprocessing`, або `sequential`.
Реалiзуй та заміряй час.

In [ ]:
# convert_grayscale iмпортовано з _note_workers.py
import time
import concurrent.futures

def download_image(url_id):
    # I/O-bound: ThreadPool - OK inline
    time.sleep(0.2)
    return f'image_{url_id}.jpg'

def read_config(file_id):
    # I/O-bound: ThreadPool - OK inline
    time.sleep(0.005)
    return {'id': file_id}

if __name__ == '__main__':
    start = time.perf_counter()
    with concurrent.futures.ThreadPoolExecutor(max_workers=10) as ex:
        results_a = list(ex.map(download_image, range(20)))
    t = time.perf_counter() - start
    print(f'Задача A (I/O): ThreadPool -> {t:.2f}s')

    # TODO: Задача B з ProcessPool
    # with concurrent.futures.ProcessPoolExecutor(max_workers=os.cpu_count()) as ex:
    #     results_b = list(ex.map(convert_grayscale, range(20)))

    # TODO: Задача C -- sequential достатньо? Заміряй!
    start = time.perf_counter()
    results_c = [read_config(i) for i in range(100)]
    t = time.perf_counter() - start
    print(f'Задача C (tiny I/O): sequential -> {t:.3f}s')

## Вправа 3: Debug Challenge (Рiвень: Advanced)

Знайди та виправ всi баги в кодi нижче. Поясни кожен баг.

In [ ]:
# worker_task iмпортовано з _note_workers.py
import multiprocessing as mp
import concurrent.futures

# BUG HUNT

# Bug 1: Missing guard
# if __name__ == '__main__':  <-- ОБОВ'ЯЗКОВО!
#     p = mp.Process(target=worker_task, args=(5,))
#     p.start(); p.join()

# Bug 2: Lambda в ProcessPool (pickle error)
# Не можна: executor.map(lambda x: x*2, range(10))
# Можна:    executor.map(worker_task, range(10))  # iменована функцiя

# Bug 3: Zombie processes
# def create_zombies():
#     procs = [mp.Process(target=worker_task, args=(i,)) for i in range(5)]
#     for p in procs: p.start()
#     return procs  # НЕМАЄ join() -> zombies!

# Bug 4: Oversubscription
# with concurrent.futures.ProcessPoolExecutor(max_workers=1000) as ex:
#     results = list(ex.map(worker_task, range(100)))

if __name__ == '__main__':
    print('Розкоментуй баги по одному, виправ i запусти.')
    print('worker_task iмпортовано з _note_workers.py')